# Préparation des données — Système de recommandation

Objectif : préparer les données pour comparer 3 approches (baseline, filtrage collaboratif, SVD), avec un split train/test adapté aux systèmes de recommandation.

In [1]:
import pandas as pd
import numpy as np

ratings = pd.read_csv('../data/ratings.csv')
movies = pd.read_csv('../data/movies.csv')

print(f"Ratings : {ratings.shape}")
ratings.head()

Ratings : (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Split train/test adapté aux systèmes de recommandation

Un split aléatoire classique (`train_test_split` sur toutes les lignes) risquerait de mettre TOUTES les notes d'un utilisateur dans le test set, rendant impossible toute recommandation pour lui (aucune donnée d'entraînement le concernant). 

**Stratégie retenue** : pour chaque utilisateur, on garde ses notes les plus anciennes en train et ses notes les plus récentes en test (approche réaliste : on prédit le futur à partir du passé) — ou plus simplement ici, on prélève un pourcentage fixe de notes de CHAQUE utilisateur pour le test, garantissant que chaque utilisateur est représenté dans les deux ensembles.

In [2]:
def train_test_split_by_user(ratings_df, test_size=0.2, min_ratings=5, random_state=42):
    """
    Split train/test qui garantit que chaque utilisateur (ayant au moins
    min_ratings notes) est représenté dans les deux ensembles.
    """
    rng = np.random.RandomState(random_state)
    train_idx = []
    test_idx = []

    for user_id, group in ratings_df.groupby('userId'):
        n = len(group)
        if n < min_ratings:
            # Trop peu de notes : tout va en train, rien en test pour cet utilisateur
            train_idx.extend(group.index.tolist())
            continue
        indices = group.index.tolist()
        rng.shuffle(indices)
        n_test = max(1, int(n * test_size))
        test_idx.extend(indices[:n_test])
        train_idx.extend(indices[n_test:])

    return ratings_df.loc[train_idx].reset_index(drop=True), ratings_df.loc[test_idx].reset_index(drop=True)

train_ratings, test_ratings = train_test_split_by_user(ratings, test_size=0.2)

print(f"Train : {train_ratings.shape}")
print(f"Test  : {test_ratings.shape}")
print(f"\nUtilisateurs dans train : {train_ratings['userId'].nunique()}")
print(f"Utilisateurs dans test  : {test_ratings['userId'].nunique()}")
print(f"Utilisateurs dans les deux : {len(set(train_ratings['userId']) & set(test_ratings['userId']))}")

Train : (80896, 4)
Test  : (19940, 4)

Utilisateurs dans train : 610
Utilisateurs dans test  : 610
Utilisateurs dans les deux : 610


## Construction de la matrice utilisateur-film (train)

Matrice creuse : lignes = utilisateurs, colonnes = films, valeurs = notes (NaN si non notée).

In [3]:
user_movie_matrix = train_ratings.pivot_table(
    index='userId', columns='movieId', values='rating'
)

print(f"Dimensions de la matrice : {user_movie_matrix.shape}")
print(f"Pourcentage de cases remplies : {(1 - user_movie_matrix.isnull().mean().mean())*100:.2f}%")

user_movie_matrix.head()

Dimensions de la matrice : (610, 8998)
Pourcentage de cases remplies : 1.47%


movieId,1,2,3,4,5,6,7,8,9,10,...,191005,193565,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Baseline : moyenne globale et moyenne par film

On prépare ici les statistiques nécessaires à la baseline la plus simple, qui servira de point de comparaison pour juger si les approches plus complexes apportent une vraie valeur ajoutée.

In [4]:
global_mean = train_ratings['rating'].mean()
movie_mean = train_ratings.groupby('movieId')['rating'].mean()
user_mean = train_ratings.groupby('userId')['rating'].mean()

print(f"Moyenne globale des notes : {global_mean:.3f}")
print(f"\nExemple - moyenne des 5 premiers films :")
print(movie_mean.head())

Moyenne globale des notes : 3.503

Exemple - moyenne des 5 premiers films :
movieId
1    3.931034
2    3.356250
3    3.292683
4    2.357143
5    3.060976
Name: rating, dtype: float64


## Sauvegarde des données préparées

In [5]:
import os
os.makedirs('../data/processed', exist_ok=True)

train_ratings.to_csv('../data/processed/train_ratings.csv', index=False)
test_ratings.to_csv('../data/processed/test_ratings.csv', index=False)

print("Fichiers sauvegardés dans data/processed/")
print(f"\nglobal_mean à réutiliser dans le prochain notebook : {global_mean:.4f}")

Fichiers sauvegardés dans data/processed/

global_mean à réutiliser dans le prochain notebook : 3.5033
